# **Atividade Prática**
<font size=3>

- **Tema:** fluxo de trabalho.
- **Prazo de entrega:** 29 de Abril.

**Envie** o notebook **executado** em formato **ipynb** pelo [formulário](https://docs.google.com/forms/d/e/1FAIpQLSfhkf8HoNNsr9WixEVVlxh8-pFK-rnXsLKN_OLRH_Tg5-5SmA/viewform?usp=sharing&ouid=111377632325147218671).

---

## **Questão:**
<font size=3>

Com base no *dataset* de [review de skincare](https://www.kaggle.com/datasets/nenamalikah/nlp-ulta-skincare-reviews), disponível no diretório $\text{dataset/}\,$, realize os seguintes passos:
1. Importe o *dataset* e verifique se existe algum valor `NaN`, usando o método `df.isnull().any()`. Caso exita, verifique a quantidade de linhas com esse problema, com `df.isnull().any().sum()`. Se forem poucas linhas de dados, remova as linhas com `df = df.dropna()`.
2. Defina o atributo `Verified_Buyer` como a variável alvo (`y`) e o atributo de *reviews* como `texts`;
3. Examine alguns poemas de `texts` a fim de definir as etapas de pré-processamento;
4. Divida o *dataframe* entre dados de treinamento e teste;
5. Transforme as categorias das variáveis `y_train` e `y_test` em representações numéricas com a classe [`LabelEncoder`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.LabelEncoder.html);
6. Com base no **_notebook_ 8**, defina o objeto do [`Pipeline`](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html) a fim de encadear todo o pré-processamento do [`ColumnTransformer`](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html) — com os dados de **textos**, **categóricos** e **numéricos** — junto ao modelo [`RandomForestClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html);
7. Encontre o melhor pré-processamento e modelo, utilizando o [`GridSearchCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html), para o hiperparâmetros `n_components` (do [`TruncatedSVD`](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.TruncatedSVD.html)) e `n_estimators` (número de árvores);
8. Realize a **avaliação** com a função [`classification_report`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html). **Escreva** em um **célula markdown** sua interpretação dos resultados.


## Resolução

### Investigação preliminar

#### Imports e parametros gerais

In [1]:
import re
import numpy as np
import pandas as pd
import nltk
from nltk.corpus   import stopwords
from nltk.corpus   import wordnet
from nltk.tokenize import word_tokenize
from nltk.stem     import WordNetLemmatizer
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, normalize

In [2]:
MIN_DF=5 # mínimo de documentos que uma palavra deve aparecer para ser considerada
MAX_DF=0.95 # máximo de documentos que uma palavra deve aparecer para ser considerada
RANDOM_STATE=42 # semente para garantir a reprodutibilidade dos resultados
TEST_SIZE=0.10 # proporção de dados para teste
SVD_COMPONENTS=25 # número de componentes para o SVD

## Configurações gerais do pandas

pd.set_option("display.max_colwidth", 500)   # não truncar texto
pd.set_option("display.max_columns", None)    # mostrar todas as colunas
pd.set_option("display.width", 120)             # deixa o pandas ajustar a largura


In [3]:
# Faz o download da lista de stop words:
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger_eng')

stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/gilcesarf/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/gilcesarf/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


#### Leitura do dataset

In [4]:
df = pd.read_csv("./dataset/skincare_reviews.csv") # estou executando no meu notebook pessoal, não no google colab.

print(df.shape)

df.head()

(4150, 10)


,Review_Title,Review_Text,Verified_Buyer,Review_Date,Review_Location,Review_Upvotes,Review_Downvotes,Product,Brand,Scrape_Date
0,Perfect,Love using this on my face while in the shower. Heats up and gives a light scrub nicely,No,15 days ago,Undisclosed,0,0,Multi-Vitamin Thermafoliant,Dermalogica,3/27/23
1,You need this,Even better than the daily microfoliant. I'm obsessed. My skin is SO MUCH smoother,No,27 days ago,Undisclosed,0,0,Multi-Vitamin Thermafoliant,Dermalogica,3/27/23
2,Clean skin,Enjoy this product so much ! I look forward to using it - really feels great.,No,2 months ago,Undisclosed,0,0,Multi-Vitamin Thermafoliant,Dermalogica,3/27/23
3,Love This Stuff!,I've never tried anything like this before and I love it. When you apply it to your face you get a little shot of warm that feels so good. The scrub seems very gritty but the only side effects I've encountered have been positive ones.,No,2 months ago,Undisclosed,0,0,Multi-Vitamin Thermafoliant,Dermalogica,3/27/23
4,This exfoliates very nicely and,"This exfoliates very nicely and gives a very smooth skin after with no irritation and no reaction to the skin . I highly recommended it, i will buy it again.",No,2 months ago,Undisclosed,0,0,Multi-Vitamin Thermafoliant,Dermalogica,3/27/23


#### Investigando nulos

In [5]:
df.isnull().any()


Review_Title        False
Review_Text          True
Verified_Buyer      False
Review_Date         False
Review_Location      True
Review_Upvotes      False
Review_Downvotes    False
Product             False
Brand               False
Scrape_Date         False
dtype: bool

In [6]:
print(f"Quantidade de linhas com valores null: {df.isnull().any(axis=1).sum().item()}")

Quantidade de linhas com valores null: 4


##### Quais são as linhas com valor nulo:

In [7]:
df[df.isnull().any(axis=1)]

,Review_Title,Review_Text,Verified_Buyer,Review_Date,Review_Location,Review_Upvotes,Review_Downvotes,Product,Brand,Scrape_Date
3397,Half full product,"The only reason I'm rating this three stars is because it's already a travel-size item, do they really needs to only fill the bottle up halfway? For how pricey this travel size bottle is, they could at least fill the whole thing up or use a smaller container, because otherwise it seems misleading.",No,2 years ago,NaN,2,0,Daily Microfoliant,Dermalogica,3/27/23
3560,I would definitely buy this product again,NaN,Yes,3 years ago,MN,0,1,Daily Microfoliant,Dermalogica,3/27/23
3684,Received a sample and loved it!,NaN,Yes,4 years ago,"Columbia, SC",0,0,Daily Microfoliant,Dermalogica,3/27/23
3686,This product works,NaN,Yes,4 years ago,"Columbia, SC",0,0,Daily Microfoliant,Dermalogica,3/27/23


##### Limpando os nulos (no mesmo dataset)

In [8]:
df.dropna(inplace=True)

##### Conferindo a limpeza

In [9]:
print(f"Quantidade de linhas com valores null: {df.isnull().any(axis=1).sum().item()}")

Quantidade de linhas com valores null: 0


#### Examinando alguns textos

In [10]:
cols_str = df.select_dtypes(include=["object", "string"]).columns
n = 3
amostra = (
    pd.concat([
        df.head(10),
        df.tail(10),
        df.sample(n=n, random_state=RANDOM_STATE),
    ])
    .drop_duplicates()  # evita repetir linhas se cair na amostra/heads/tails
)
for i, row in amostra.iterrows():
    print(f"Review (Title + Text): {row["Review_Title"]} {row["Review_Text"]}")


Review (Title + Text): Perfect Love using this on my face while in the shower. Heats up and gives a light scrub nicely
Review (Title + Text): You need this Even better than the daily microfoliant. I'm obsessed. My skin is SO MUCH smoother
Review (Title + Text): Clean skin Enjoy this product so much ! I look forward to using it - really feels great.
Review (Title + Text): Love This Stuff! I've never tried anything like this before and I love it. When you apply it to your face you get a little shot of warm that feels so good. The scrub seems very gritty but the only side effects I've encountered have been positive ones.
Review (Title + Text): This exfoliates very nicely and This exfoliates very nicely and gives a very smooth skin after with no irritation and no reaction to the skin . I highly recommended it, i will buy it again.
Review (Title + Text): Seriously nice scrub! Love that you can use it wet and dry, you can control how abrasive it is. Leaves your face soft and drenched in vita

#### Definindo uma função de pré-processamento

In [14]:
lemmatizer = WordNetLemmatizer()

def penn_to_wordnet_pos(tag: str):
    """
    Converte uma tag POS (classe gramatical) do Penn Treebank (saída do nltk.pos_tag) 
    para uma categoria do WordNet compatível com o WordNetLemmatizer.
    Args:
        tag (str): Tag POS do Penn Treebank, por exemplo "NN", "VB", "JJ", "RB".
    Returns:
        str | None: Uma das categorias do WordNet (wordnet.NOUN, wordnet.VERB, wordnet.ADJ, wordnet.ADV)
        ou None quando a tag não pertence a {J, V, N, R}.
    Notes:
        - Mapeamento:
          - J* -> ADJ
          - V* -> VERB
          - N* -> NOUN
          - R* -> ADV
        - Retornar None é uma abordagem conservadora: se o POS não for reconhecido, não força
          a lematização com uma classe incorreta.
    """
    if tag.startswith("J"):
        return wordnet.ADJ
    if tag.startswith("V"):
        return wordnet.VERB
    if tag.startswith("N"):
        return wordnet.NOUN
    if tag.startswith("R"):
        return wordnet.ADV
    return None

def preprocess_text(text, lemma=False):
    """
    Normaliza um texto com limpeza básica, remoção de stopwords e lematização opcional com POS-tagging.
    Etapas:
        - Converte para minúsculas.
        - Remove pontuação/caracteres não alfanuméricos, dígitos e espaços extras.
        - Tokeniza por espaços.
        - Remove stopwords (variável externa stop_words).
        - Se lemma=True, aplica POS-tagging (nltk.pos_tag), mapeia POS para WordNet e lematiza
          com WordNetLemmatizer de forma conservadora (só lematiza quando há POS mapeável).
    Args:
        text (str): Texto de entrada.
        lemma (bool): Se True, aplica lematização com POS. Se False, não lematiza.
    Returns:
        str: Texto processado, com tokens separados por um único espaço.
    Dependencies:
        - re, nltk
        - stop_words (iterável/set definido externamente)
        - lemmatizer (WordNetLemmatizer definido externamente)
        - penn_to_wordnet_pos(tag)
    """
    text = text.lower()                               # converte para minúsculas

    # mantém apóstrofo para contrações (don't, i'm, you're) e remove o resto
    text = re.sub(r"[^a-z0-9'\s]", " ", text)          # substitui caracteres não alfanuméricos por espaços
    text = re.sub(r"\s+", " ", text)                  # substitui múltiplos espaços por um único espaço
    text = re.sub(r"\d+", "", text)                   # remove dígitos
    text = text.strip()                               # remove espaços em branco no início e no final

    tokens = word_tokenize(text)                      # tokeniza usando nltk pois ele lida melhor com contrações

    tokens = [t for t in tokens
              if re.search(r"[a-z]", t)]              # remove tokens que não possuem letras
              
    # tokens = [token for token in tokens
    #           if token not in stop_words]             # remove stop words

    if lemma:
        tagged = nltk.pos_tag(tokens)
        tokens = [# aplica lematização se houver POS mapeável
                  (lemmatizer.lemmatize(token, pos=wn_pos) if wn_pos else token)  
                  for token, tag in tagged
                  # truque para invocar penn_to_wordnet_pos e nomear resultado como wn_pos
                  for wn_pos in [penn_to_wordnet_pos(tag)]]

    return " ".join(tokens)

In [15]:

for i, row in amostra.iterrows():
    print("Review (Title + Text):", preprocess_text(f"{row['Review_Title']} {row['Review_Text']}"))


Review (Title + Text): perfect love using this on my face while in the shower heats up and gives a light scrub nicely
Review (Title + Text): you need this even better than the daily microfoliant i 'm obsessed my skin is so much smoother
Review (Title + Text): clean skin enjoy this product so much i look forward to using it really feels great
Review (Title + Text): love this stuff i 've never tried anything like this before and i love it when you apply it to your face you get a little shot of warm that feels so good the scrub seems very gritty but the only side effects i 've encountered have been positive ones
Review (Title + Text): this exfoliates very nicely and this exfoliates very nicely and gives a very smooth skin after with no irritation and no reaction to the skin i highly recommended it i will buy it again
Review (Title + Text): seriously nice scrub love that you can use it wet and dry you can control how abrasive it is leaves your face soft and drenched in vitamins
Review (Tit

In [16]:

for i, row in amostra.iterrows():
    print("Review (Title + Text):", preprocess_text(f"{row['Review_Title']} {row['Review_Text']}", lemma=True))


Review (Title + Text): perfect love use this on my face while in the shower heat up and give a light scrub nicely
Review (Title + Text): you need this even good than the daily microfoliant i 'm obsessed my skin be so much smoother
Review (Title + Text): clean skin enjoy this product so much i look forward to use it really feel great
Review (Title + Text): love this stuff i 've never try anything like this before and i love it when you apply it to your face you get a little shot of warm that feel so good the scrub seem very gritty but the only side effect i 've encounter have be positive one
Review (Title + Text): this exfoliate very nicely and this exfoliate very nicely and give a very smooth skin after with no irritation and no reaction to the skin i highly recommend it i will buy it again
Review (Title + Text): seriously nice scrub love that you can use it wet and dry you can control how abrasive it be leave your face soft and drench in vitamin
Review (Title + Text): absolutely love 